# llm-memlab Colab Quickstart

This notebook is designed for beginners: install the library, configure paths, download a Hugging Face model, check the runtime, run the memory-first workflow, and open the dashboard.

Fast path: enable GPU if available (`Runtime > Change runtime type > GPU`), then click **Runtime > Run all**. The default model is intentionally tiny so the first smoke test is easy to run. After it works, change `MODEL_PRESET` to try a larger model.

## 1. Setup - Install the library

This cell clones the repo and installs `llm-memlab` with the basic dependencies. Run it first whenever you start a fresh runtime.

## Before You Start - Pick a workload size

Start with the defaults. They are fast, cheap, and confirm that the environment works. After the dashboard is generated, increase one setting at a time.

- Beginner / quick smoke test: `MODEL_PRESET = "tiny"`, `LLM_MEMLAB_TOKENS = "1"`
- Small real model: `MODEL_PRESET = "tinyllama"`
- Qwen3 test: use a GPU runtime and set `MODEL_PRESET = "qwen3"`
- Private or gated model: set `HF_TOKEN`, then use `custom`

In [ ]:
!git clone https://github.com/thhanabun/LLMOptimization.git
%cd LLMOptimization
!python -m pip install -e .[torch,transformers]

## 2. Configure - Paths and runtime settings

This is the main beginner-friendly control cell. Most users only need to edit `LLM_MEMLAB_TOKENS`, `LLM_MEMLAB_CACHE`, or `LLM_MEMLAB_PROMPT`. Models and reports are stored under `/content/hf_models` and `/content/llm_memlab_outputs`.

In [ ]:
import os

# Beginner knobs: these are enough for most first runs.
os.environ.setdefault("LLM_MEMLAB_MODEL_ROOT", "/content/hf_models")
os.environ.setdefault("LLM_MEMLAB_OUT_DIR", "/content/llm_memlab_outputs")
os.environ.setdefault("LLM_MEMLAB_DEVICE", "auto")  # auto | cpu | cuda
os.environ.setdefault("LLM_MEMLAB_DTYPE", "auto")    # auto | float16 | bfloat16 | float32
os.environ.setdefault("LLM_MEMLAB_CACHE", "paged")   # paged | quantized
os.environ.setdefault("LLM_MEMLAB_TOKENS", "1")      # increase to 8/16 after the smoke test passes
os.environ.setdefault("LLM_MEMLAB_PROMPT", "hello")

for key in [
    "LLM_MEMLAB_MODEL_ROOT",
    "LLM_MEMLAB_OUT_DIR",
    "LLM_MEMLAB_DEVICE",
    "LLM_MEMLAB_DTYPE",
    "LLM_MEMLAB_CACHE",
    "LLM_MEMLAB_TOKENS",
    "LLM_MEMLAB_PROMPT",
]:
    print(f"{key}={os.environ[key]}")

## 3. Load Model - Download from Hugging Face

Change only `MODEL_PRESET` for the common cases:

- `tiny` = default, very small, best for smoke tests and low-memory runtimes
- `tinyllama` = a small real model; slower, but still reasonable on cloud GPUs
- `qwen3` = Qwen3 1.7B; use a GPU runtime and expect more disk/RAM usage
- `custom` = use your own repo id from `CUSTOM_MODEL_ID`

For gated/private models, set `HF_TOKEN` before running this cell.

In [ ]:
from huggingface_hub import snapshot_download

MODEL_PRESET = "tiny"  # tiny | tinyllama | qwen3 | custom
CUSTOM_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

MODEL_IDS = {
    "tiny": "hf-internal-testing/tiny-random-LlamaForCausalLM",
    "tinyllama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "qwen3": "Qwen/Qwen3-1.7B",
    "custom": CUSTOM_MODEL_ID,
}

HF_MODEL_ID = MODEL_IDS[MODEL_PRESET]
HF_TOKEN = os.environ.get("HF_TOKEN")
local_dir = os.path.join(os.environ["LLM_MEMLAB_MODEL_ROOT"], HF_MODEL_ID.replace("/", "__"))

print("Downloading or reusing cached model:", HF_MODEL_ID)
model_path = snapshot_download(repo_id=HF_MODEL_ID, local_dir=local_dir, token=HF_TOKEN)
os.environ["LLM_MEMLAB_MODEL"] = model_path
print("Using model:", HF_MODEL_ID)
print("Local path:", model_path)

## 4. Debug - Check the runtime

This is not the main benchmark yet. It checks whether Torch, CUDA, Triton, and vLLM are visible, then prints a rough memory estimate. If CUDA is not available, the notebook can still run on CPU, but it will be slower, so keep `MODEL_PRESET = "tiny"` first.

In [ ]:
!python -m llm_memlab backend-demo
!python -m llm_memlab estimate --preset 7b-like --seq 2048 --batch 1 --training inference

## 5. Optimize / Benchmark - Run the memory-first workflow

This cell calls the real library workflow through `examples/cloud_smoke.py` and writes benchmark artifacts. It will:

- check backend availability and memory estimates again
- run an HF generate baseline when a model is available
- run the llm-memlab memory-first HF path
- try a vLLM baseline when the environment supports it
- write JSON, CSV, and HTML dashboard files

After the first run passes, increase `LLM_MEMLAB_TOKENS` to 8 or 16 for a more realistic check.

In [ ]:
!python examples/cloud_smoke.py \
  --model "$LLM_MEMLAB_MODEL" \
  --model-root "$LLM_MEMLAB_MODEL_ROOT" \
  --prompt "$LLM_MEMLAB_PROMPT" \
  --tokens $LLM_MEMLAB_TOKENS \
  --device $LLM_MEMLAB_DEVICE \
  --dtype $LLM_MEMLAB_DTYPE \
  --cache $LLM_MEMLAB_CACHE \
  --out-dir "$LLM_MEMLAB_OUT_DIR"

## 6. View Report - Open the dashboard

The dashboard shows latency, quality, memory, selected backend, and fallback reasons. The HTML file is written to `/content/llm_memlab_outputs/cloud_dashboard.html`, so you can reopen or download it after the run.

In [ ]:
from IPython.display import HTML, display

out_dir = os.environ["LLM_MEMLAB_OUT_DIR"]
dashboard = os.path.join(out_dir, "cloud_dashboard.html")
print("Dashboard:", dashboard)
if os.path.exists(dashboard):
    display(HTML(open(dashboard, encoding="utf-8").read()))
else:
    print("Dashboard not found yet. Run section 5 first.")

## Troubleshooting

- Clone/download fails: make sure the Colab runtime has internet access.
- CUDA is missing: enable GPU at `Runtime > Change runtime type > GPU`, then restart the runtime/kernel.
- RAM/VRAM is too small: use `MODEL_PRESET = "tiny"`, keep `LLM_MEMLAB_TOKENS = "1"`, or set `LLM_MEMLAB_DEVICE = "cpu"` for a smoke test.
- Gated model requires access: request access on Hugging Face and set `HF_TOKEN`.
- vLLM is unavailable: that is okay; the dashboard records the fallback reason and still compares the HF and memory-first paths.
- Dashboard is missing: run section 5 first, then check `LLM_MEMLAB_OUT_DIR`.